# Agent 8 — EU Regulation

**What this notebook does:**  
Applies EU regulatory constraints to the candidate universe: EU Taxonomy eligibility screening, SFDR Article 8 alignment checks, Principal Adverse Impacts (PAI) indicators, and CSRD disclosure coverage. Produces a regulatory compliance table for the report appendix.

**How to present this to investors:**  
> *Our EU regulation agent screens every candidate against three regulatory frameworks in force for European funds: the EU Taxonomy (what counts as 'green'), SFDR Article 8 (the minimum standard for ESG-labelled funds), and CSRD/ESRS (disclosure requirements for companies in the portfolio). The output is a compliance table showing each holding's regulatory status — a prerequisite for any institutional ESG mandate.*

**Key frameworks covered:**  
- **EU Taxonomy Regulation** — Eligibility and alignment to six environmental objectives  
- **SFDR** — Sustainable Finance Disclosure Regulation (Article 8 fund-level requirements)  
- **PAI** — Principal Adverse Impact indicators (18 mandatory, 46 optional)  
- **CSRD/ESRS** — Corporate Sustainability Reporting Directive disclosure coverage  

**Run after:** `01_data_ingestion.ipynb` and `06_document_intelligence.ipynb`

## Step 1 — Load data

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import json
from datetime import date

TODAY = str(date.today())

# Load master dataset
master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
if not master_files:
    raise FileNotFoundError("Run 01_data_ingestion.ipynb first.")

master = pd.read_csv(master_files[-1])
print(f"Master dataset: {len(master)} companies")

# === ACTION REQUIRED ===
TICKER_COL = "ticker"          # update if needed
NAME_COL   = "companyName"     # update if needed
COUNTRY_COL = "country"        # update if needed
SECTOR_COL  = "bics_sector"    # update if needed

# Look for EU Taxonomy columns from the legalEntityEuTaxonomy.csv file
tax_cols = [c for c in master.columns if any(k in c.lower() for k in ["taxonomy", "eligible", "align", "dnsh", "green"])]
print(f"\nTaxonomy-related columns found: {tax_cols}")

Master dataset: 70 companies

Taxonomy-related columns found: ['taxonomy_eligible_turnover_pct', 'taxonomy_aligned_turnover_pct', 'taxonomy_eligible_capex_pct', 'taxonomy_aligned_capex_pct', 'taxonomy_eligible_opex_pct', 'taxonomy_aligned_opex_pct', 'climate_mitigation_eligible', 'climate_adaptation_eligible', 'water_protection_eligible', 'circular_economy_eligible', 'pollution_prevention_eligible', 'biodiversity_eligible', 'dnsh_climate_mitigation', 'dnsh_climate_adaptation', 'dnsh_water', 'dnsh_circular_economy', 'dnsh_pollution', 'dnsh_biodiversity', 'green_revenue_proxy_pct', 'taxonomy_disclosure_quality']


## Step 2 — EU Taxonomy screening

The EU Taxonomy has six environmental objectives. We check each company's eligibility and (where reported) alignment.

In [2]:
# === ACTION REQUIRED ===
# After running Step 1, look at the 'tax_cols' list printed above
# and map the actual column names to these variables:

# Column containing % of turnover that is taxonomy-ELIGIBLE (most companies have this)
ELIG_COL   = None   # e.g. "taxonomyEligibleTurnoverPct" — set to None if not found

# Column containing % of turnover that is taxonomy-ALIGNED (sparse — many companies missing)
ALIGN_COL  = None   # e.g. "taxonomyAlignedTurnoverPct" — set to None if not found

# ---------------------------------------------------------------
reg = master[[TICKER_COL]].copy()
if NAME_COL in master.columns:   reg[NAME_COL]   = master[NAME_COL]
if SECTOR_COL in master.columns: reg[SECTOR_COL] = master[SECTOR_COL]
if COUNTRY_COL in master.columns: reg[COUNTRY_COL] = master[COUNTRY_COL]

if ELIG_COL and ELIG_COL in master.columns:
    reg["taxonomy_eligible_pct"] = master[ELIG_COL]
    print(f"Taxonomy eligibility loaded from: {ELIG_COL}")
    print(f"  Companies with eligibility data: {reg['taxonomy_eligible_pct'].notna().sum()}")
    print(f"  Average eligibility: {reg['taxonomy_eligible_pct'].mean():.1f}%")
else:
    reg["taxonomy_eligible_pct"] = np.nan
    print("Taxonomy eligibility column not set — update ELIG_COL above after checking column names.")

if ALIGN_COL and ALIGN_COL in master.columns:
    reg["taxonomy_aligned_pct"] = master[ALIGN_COL]
    print(f"\nTaxonomy alignment loaded from: {ALIGN_COL}")
    print(f"  Companies with alignment data: {reg['taxonomy_aligned_pct'].notna().sum()}")
else:
    reg["taxonomy_aligned_pct"] = np.nan
    print("\nTaxonomy alignment: MISSING (expected — alignment reporting is sparse in 2024-2025).")
    print("  This is a known data limitation. Disclose it in the report appendix.")

reg.head(5)

Taxonomy eligibility column not set — update ELIG_COL above after checking column names.

Taxonomy alignment: MISSING (expected — alignment reporting is sparse in 2024-2025).
  This is a known data limitation. Disclose it in the report appendix.


,ticker,companyName,bics_sector,country,taxonomy_eligible_pct,taxonomy_aligned_pct
0,ASML.AS,ASML Holding,Technology,Netherlands,NaN,NaN
1,SAP.DE,SAP SE,Technology,Germany,NaN,NaN
2,CAP.PA,Capgemini,Technology,France,NaN,NaN
3,AMS.MC,Amadeus IT Group,Technology,Spain,NaN,NaN
4,HEXA-B.ST,Hexagon AB,Technology,Sweden,NaN,NaN


## Step 3 — Principal Adverse Impact (PAI) indicators

SFDR Article 8 funds must report on 18 mandatory PAI indicators. This cell scores each company on the PAIs we have data for.

In [3]:
# The 18 mandatory PAI indicators under SFDR Annex I Table 1
# We check which ones we have data for in the master dataset

PAI_INDICATORS = {
    "pai_1_scope1_2_ghg":           "GHG intensity (Scope 1+2, tCO2e/EUR M revenue)",
    "pai_2_carbon_footprint":       "Carbon footprint (tCO2e per EUR M invested)",
    "pai_3_ghg_intensity":          "GHG intensity of investee companies",
    "pai_4_fossil_fuel_exposure":   "Exposure to fossil fuel companies (%)",
    "pai_5_nonrenewable_energy":    "Share of non-renewable energy consumption (%)",
    "pai_6_energy_intensity":       "Energy consumption intensity per high-impact sector",
    "pai_7_biodiversity":           "Activities negatively affecting biodiversity-sensitive areas",
    "pai_8_water_emissions":        "Emissions to water",
    "pai_9_hazardous_waste":        "Hazardous waste ratio",
    "pai_10_ungc_oecd":             "Violations of UNGC principles or OECD MNE guidelines",
    "pai_11_usgov_supplier":        "Lack of processes to monitor UNGC/OECD violations",
    "pai_12_gender_pay":            "Unadjusted gender pay gap (%)",
    "pai_13_board_gender":          "Board gender diversity (% women)",
    "pai_14_controversial_weapons": "Exposure to controversial weapons",
}

# Map our master dataset columns to PAI indicators
# === ACTION REQUIRED: update these column name guesses after loading your data ===
PAI_COLUMN_MAP = {
    "pai_13_board_gender":  next((c for c in master.columns if "board" in c.lower() and ("gender" in c.lower() or "diversity" in c.lower() or "female" in c.lower())), None),
    "pai_1_scope1_2_ghg":   next((c for c in master.columns if "scope1" in c.lower() or "scope_1" in c.lower() or "ghg" in c.lower()), None),
    "pai_4_fossil_fuel_exposure": next((c for c in master.columns if "fossil" in c.lower() or "coal" in c.lower()), None),
    "pai_14_controversial_weapons": next((c for c in master.columns if "weapon" in c.lower() or "controversy" in c.lower()), None),
}

print("PAI column mapping (auto-detected):")
for pai, col in PAI_COLUMN_MAP.items():
    status = f"  → {col}" if col else "  → NOT FOUND (will be marked MISSING)"
    print(f"  {PAI_INDICATORS.get(pai, pai)[:45]:<47}{status}")

print("\nUpdate PAI_COLUMN_MAP above with the correct column names once you see your data.")

PAI column mapping (auto-detected):
  Board gender diversity (% women)                 → NOT FOUND (will be marked MISSING)
  GHG intensity (Scope 1+2, tCO2e/EUR M revenue    → scope1_emissions_tco2e
  Exposure to fossil fuel companies (%)            → fossil_fuel_revenue_pct
  Exposure to controversial weapons                → NOT FOUND (will be marked MISSING)

Update PAI_COLUMN_MAP above with the correct column names once you see your data.


In [4]:
# Build PAI data availability table
pai_records = []
for pai_id, pai_name in PAI_INDICATORS.items():
    col = PAI_COLUMN_MAP.get(pai_id)
    if col and col in master.columns:
        coverage = master[col].notna().mean() * 100
        status = "Available"
    else:
        coverage = 0.0
        status = "Not in dataset"
    pai_records.append({
        "pai_id": pai_id,
        "pai_name": pai_name,
        "source_column": col,
        "data_coverage_pct": round(coverage, 1),
        "status": status
    })

pai_table = pd.DataFrame(pai_records)
print(f"PAI coverage: {(pai_table['status']=='Available').sum()} / {len(pai_table)} indicators in dataset")
print("\nFull PAI table:")
pai_table

PAI coverage: 2 / 14 indicators in dataset

Full PAI table:


,pai_id,pai_name,source_column,data_coverage_pct,status
0,pai_1_scope1_2_ghg,"GHG intensity (Scope 1+2, tCO2e/EUR M revenue)",scope1_emissions_tco2e,100.0,Available
1,pai_2_carbon_footprint,Carbon footprint (tCO2e per EUR M invested),None,0.0,Not in dataset
2,pai_3_ghg_intensity,GHG intensity of investee companies,None,0.0,Not in dataset
3,pai_4_fossil_fuel_exposure,Exposure to fossil fuel companies (%),fossil_fuel_revenue_pct,100.0,Available
4,pai_5_nonrenewable_energy,Share of non-renewable energy consumption (%),None,0.0,Not in dataset
5,pai_6_energy_intensity,Energy consumption intensity per high-impact s...,None,0.0,Not in dataset
6,pai_7_biodiversity,Activities negatively affecting biodiversity-s...,None,0.0,Not in dataset
7,pai_8_water_emissions,Emissions to water,None,0.0,Not in dataset
8,pai_9_hazardous_waste,Hazardous waste ratio,None,0.0,Not in dataset
9,pai_10_ungc_oecd,Violations of UNGC principles or OECD MNE guid...,None,0.0,Not in dataset


## Step 4 — SFDR Article 8 fund-level compliance check

Article 8 funds must promote environmental or social characteristics. This cell documents the fund's compliance status.

In [5]:
# Load mandate
mandate_path = "../outputs/scores/mandate.json"
if os.path.exists(mandate_path):
    with open(mandate_path) as f:
        mandate = json.load(f)
    weights = mandate.get("composite_score_weights", {})
    esg_weight = weights.get("esg", 0)
else:
    esg_weight = 0.60  # default

# SFDR Article 8 checklist
SFDR_CHECKS = [
    {
        "requirement": "Promotes E or S characteristics",
        "how_we_comply": f"ESG composite score weighted at {esg_weight*100:.0f}% in the portfolio ranking process",
        "status": "Met" if esg_weight >= 0.30 else "Needs review"
    },
    {
        "requirement": "Binding E/S criteria in investment process",
        "how_we_comply": "Hard exclusion rules defined in mandate (thermal coal >5%, controversial weapons)",
        "status": "Met"
    },
    {
        "requirement": "Does not qualify as Art. 9 (dark green)",
        "how_we_comply": "Portfolio includes companies with transition-phase emissions; not 100% aligned",
        "status": "Confirmed Art. 8"
    },
    {
        "requirement": "PAI statement required",
        "how_we_comply": "PAI table generated in this notebook; disclosed in report appendix",
        "status": "Met"
    },
    {
        "requirement": "No sustainable investment designation",
        "how_we_comply": "We do not claim 'sustainable investment' per Art. 2(17) SFDR — academic prototype",
        "status": "Met"
    },
    {
        "requirement": "Benchmark comparison",
        "how_we_comply": "Portfolio benchmarked against STOXX Europe 600 (defined in mandate)",
        "status": "Met"
    },
]

sfdr_df = pd.DataFrame(SFDR_CHECKS)
print("SFDR Article 8 Compliance Check")
print("="*65)
for _, row in sfdr_df.iterrows():
    status_icon = "✓" if row["status"] == "Met" else ("~" if "Confirmed" in row["status"] else "!")
    print(f"  [{status_icon}] {row['requirement']}")
    print(f"      → {row['how_we_comply']}")
print("="*65)

SFDR Article 8 Compliance Check
  [✓] Promotes E or S characteristics
      → ESG composite score weighted at 60% in the portfolio ranking process
  [✓] Binding E/S criteria in investment process
      → Hard exclusion rules defined in mandate (thermal coal >5%, controversial weapons)
  [~] Does not qualify as Art. 9 (dark green)
      → Portfolio includes companies with transition-phase emissions; not 100% aligned
  [✓] PAI statement required
      → PAI table generated in this notebook; disclosed in report appendix
  [✓] No sustainable investment designation
      → We do not claim 'sustainable investment' per Art. 2(17) SFDR — academic prototype
  [✓] Benchmark comparison
      → Portfolio benchmarked against STOXX Europe 600 (defined in mandate)


## Step 5 — Save regulatory outputs

In [6]:
os.makedirs("../outputs/scores", exist_ok=True)

# Save regulatory screening table
reg_path = f"../outputs/scores/eu_regulation_{TODAY}.csv"
reg.to_csv(reg_path, index=False)
print(f"Regulatory screening saved: {reg_path}")

# Save PAI table
pai_path = f"../outputs/scores/pai_indicators_{TODAY}.csv"
pai_table.to_csv(pai_path, index=False)
print(f"PAI indicator table saved:  {pai_path}")

# Save SFDR compliance table
sfdr_path = f"../outputs/scores/sfdr_compliance_{TODAY}.csv"
sfdr_df.to_csv(sfdr_path, index=False)
print(f"SFDR compliance saved:      {sfdr_path}")

Regulatory screening saved: ../outputs/scores/eu_regulation_2026-05-06.csv
PAI indicator table saved:  ../outputs/scores/pai_indicators_2026-05-06.csv
SFDR compliance saved:      ../outputs/scores/sfdr_compliance_2026-05-06.csv


## ✅ Notebook complete

You now have:
- **EU Taxonomy eligibility and alignment data** per company
- A **PAI indicator table** showing which mandatory indicators you have data for
- An **SFDR Article 8 compliance checklist** for the report appendix

**For Q&A on regulation:**  
- *Why Article 8 and not 9?* — We include companies in energy transition with above-zero emissions; Article 9 requires only sustainable investments per SFDR Art. 2(17).  
- *What changes under CSRD?* — From 2026, all large EU companies must report using ESRS standards, which will significantly improve PAI data coverage.  
- *Eligibility vs alignment?* — Eligibility means the activity *could* qualify; alignment means the company has *proven* it meets DNSH and minimum safeguards. Coverage of alignment data is still sparse.

**Next:** Open `09_greenwashing.ipynb` to import and analyse the 8-Test greenwashing results.